In [0]:
%run /Repos/dung_nguyen_hoang@mfcgd.com/Utilities/Functions

In [0]:
import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T
import pyspark.sql.window as Window

In [0]:
# Load the external table/view
# There are 2,572,980 records total
df = spark.table("vn_aa_reports.gp_mim_lifeco_sequence")

# Load APE from inforce LifeCo customers
df_ape = spark.sql("""
SELECT  client_id as lifeco_client_id, SUM(ape_usd) AS ape_usd
FROM    vn_aa_reports.lifeco_policy_data
WHERE   ape_usd > 0
GROUP BY
        client_id                   
""")

# Add additional flags
df = df.withColumn(
    "mcv", 
    F.when(df["f_mcv"] == 1, "Y")
    .otherwise("N")
).withColumn(
    "age_group", 
    F.when(df["age"] < 25, "<25")
    .when((df["age"] >= 25) & (df["age"] < 35), "25-35")
    .when((df["age"] >= 35) & (df["age"] < 45), "35-45")
    .when((df["age"] >= 45) & (df["age"] < 55), "45-55")
    .when((df["age"] >= 55) & (df["age"] < 65), "55-65")
    .otherwise(">65")
).withColumn(
    "lifeco_ind",
    F.when(
        F.coalesce(F.col("saving_policy_cnt"), F.lit(0)) +
        F.coalesce(F.col("income_policy_cnt"), F.lit(0)) +
        F.coalesce(F.col("hospital_policy_cnt"), F.lit(0)) +
        F.coalesce(F.col("life_protection_policy_cnt"), F.lit(0)) +
        F.coalesce(F.col("critical_illness_policy_cnt"), F.lit(0)) +
        F.coalesce(F.col("investment_policy_cnt"), F.lit(0)) +
        F.coalesce(F.col("premium_waiver_policy_cnt"), F.lit(0)) +
        F.coalesce(F.col("disability_policy_cnt"), F.lit(0)) +
        F.coalesce(F.col("juvenile_policy_cnt"), F.lit(0)) +
        F.coalesce(F.col("accident_policy_cnt"), F.lit(0)) +
        F.coalesce(F.col("others_policy_cnt"), F.lit(0)) > 0, 
        "Y"
    ).otherwise("N")
).withColumn(
    "mim_ind",
    F.when(F.col("wam_client_id").isNotNull(), "Y")
    .otherwise("N")
).withColumn(
    "gp_ind",
    F.when(F.col("gp_aum_usd").isNotNull(), "Y")
    .otherwise("N")
)

df = df.withColumn(
    "multiple_prod_ind",
    F.when(
        ((df["lifeco_ind"] == "Y") & (df["mim_ind"] == "Y")) |
        ((df["mim_ind"] == "Y") & (df["gp_ind"] == "Y")) |
        ((df["gp_ind"] == "Y") & (df["lifeco_ind"] == "Y")),
        F.lit(1)
    ).otherwise(0)
).fillna(
    {"age_group": "Missing"}
)

df = df.join(
    df_ape,
    on="lifeco_client_id",
    how="left"
).fillna(
    {"ape_usd": 0}
)
# print(df.count())
# check_dup(df, "lifeco_client_id")

In [0]:
# 1) Consider just customers who have multiple relationships
# There are 425,551 total, which is 16.5% of total
df_multiple_relationships = df.filter(
    F.col("multiple_prod_ind") == 1
)
# record_count = df_multiple_relationships.count()
# print(record_count)
# check_dup(df_multiple_relationships, "lifeco_client_id")

# Create purchase sequence column
df_multiple_relationships = df_multiple_relationships.withColumn(
    "purchase_sequence",
    F.when(
        (df_multiple_relationships["first_inow_submit_date"].isNotNull()) &
        (df_multiple_relationships["earliest_gp_client_date"].isNotNull()) &
        (df_multiple_relationships["earliest_wam_client_date"].isNotNull()),
        F.when(
            df_multiple_relationships["first_inow_submit_date"] < df_multiple_relationships["earliest_gp_client_date"],
            F.when(
                df_multiple_relationships["earliest_gp_client_date"] < df_multiple_relationships["earliest_wam_client_date"],
                F.concat_ws(" -> ", F.lit("LifeCo"), F.lit("GP"), F.lit("MIM"))
            ).otherwise(
                F.concat_ws(" -> ", F.lit("LifeCo"), F.lit("MIM"), F.lit("GP"))
            )
        ).otherwise(
            F.concat_ws(" -> ", F.lit("GP"), F.lit("LifeCo"), F.lit("MIM"))
        )
    )
    .when(
        (df_multiple_relationships["first_inow_submit_date"].isNotNull()) &
        (df_multiple_relationships["earliest_wam_client_date"].isNotNull()),
        F.when(
            df_multiple_relationships["first_inow_submit_date"] < df_multiple_relationships["earliest_wam_client_date"],
            F.concat_ws(" -> ", F.lit("LifeCo"), F.lit("MIM"))
        ).otherwise(
            F.concat_ws(" -> ", F.lit("MIM"), F.lit("LifeCo"))
        )
    )
    .when(
        (df_multiple_relationships["first_inow_submit_date"].isNotNull()) &
        (df_multiple_relationships["earliest_gp_client_date"].isNotNull()),
        F.when(
            df_multiple_relationships["first_inow_submit_date"] < df_multiple_relationships["earliest_gp_client_date"],
            F.concat_ws(" -> ", F.lit("LifeCo"), F.lit("GP"))
        ).otherwise(
            F.concat_ws(" -> ", F.lit("GP"), F.lit("LifeCo"))
        )
    )
    .when(
        (df_multiple_relationships["earliest_wam_client_date"].isNotNull()) &
        (df_multiple_relationships["earliest_gp_client_date"].isNotNull()),
        F.when(
            df_multiple_relationships["earliest_wam_client_date"] < df_multiple_relationships["earliest_gp_client_date"],
            F.concat_ws(" -> ", F.lit("MIM"), F.lit("GP"))
        ).otherwise(
            F.concat_ws(" -> ", F.lit("GP"), F.lit("MIM"))
        )
    )
    .otherwise(F.lit("Unknown"))
)

# 2) Purchase journey
# a. Customers starting with Lifeco
lifeco_start = df_multiple_relationships.filter(
    df["first_inow_submit_date"].isNotNull() &
    ((df["lifeco_ind"] == "Y"))
)
lifeco_start = lifeco_start.withColumn(
    "months_lifeco_to_other_products",
    F.months_between(
        F.least(df["earliest_wam_client_date"], df["earliest_gp_client_date"]),
        df["first_inow_submit_date"]
    )
).withColumn(
    "months_lifeco_to_other_products_cat",
    F.when(F.col("months_lifeco_to_other_products") <= 2, "01. <= 2mth")
    .when((F.col("months_lifeco_to_other_products") > 2) & (F.col("months_lifeco_to_other_products") <= 6), "02. 3 - 6mth")
    .when((F.col("months_lifeco_to_other_products") > 6) & (F.col("months_lifeco_to_other_products") <= 12), "03. 7 - 12mth")
    .when((F.col("months_lifeco_to_other_products") > 12) & (F.col("months_lifeco_to_other_products") <= 24), "04. 1 - 2yr")
    .when((F.col("months_lifeco_to_other_products") > 24) & (F.col("months_lifeco_to_other_products") <= 36), "05. 2 - 3yr")
    .when((F.col("months_lifeco_to_other_products") > 36) & (F.col("months_lifeco_to_other_products") <= 60), "06. 3 - 5yr")
    .when((F.col("months_lifeco_to_other_products") > 60) & (F.col("months_lifeco_to_other_products") <= 84), "07. 5 - 7yr")
    .when((F.col("months_lifeco_to_other_products") > 84) & (F.col("months_lifeco_to_other_products") <=120), "08. 7 - 10yr")
    .when((F.col("months_lifeco_to_other_products") >120) & (F.col("months_lifeco_to_other_products") <=180), "09. 10 - 15yr")
    .when((F.col("months_lifeco_to_other_products") >180) & (F.col("months_lifeco_to_other_products") <=240), "10. 15 - 20yr")
    .otherwise("11. > 20yr")
)

# Exclude customers who did not buy Lifeco first
lifeco_start_summary = lifeco_start.filter(
    F.col("months_lifeco_to_other_products") > 0
).groupBy(
    "months_lifeco_to_other_products_cat"
).agg(
    F.count("lifeco_client_id").alias("count"),
    F.avg("months_lifeco_to_other_products").alias("avg_months_lifeco_to_other_products"),
    F.sum("ape_usd").alias("total_ape")
).orderBy(
    "months_lifeco_to_other_products_cat"
)

# b. Customers starting with MIM
mim_start = df_multiple_relationships.filter(
    df["earliest_wam_client_date"].isNotNull()
)
mim_start = mim_start.withColumn(
    "months_wam_to_other_products",
    F.months_between(
        F.least(df["first_inow_submit_date"], df["earliest_gp_client_date"]),
        df["earliest_wam_client_date"]
    )
).withColumn(
    "months_wam_to_other_products_cat",
    F.when(F.col("months_wam_to_other_products") <= 2, "01. <= 2mth")
    .when((F.col("months_wam_to_other_products") > 2) & (F.col("months_wam_to_other_products") <= 6), "02. 3 - 6mth")
    .when((F.col("months_wam_to_other_products") > 6) & (F.col("months_wam_to_other_products") <= 12), "03. 7 - 12mth")
    .when((F.col("months_wam_to_other_products") > 12) & (F.col("months_wam_to_other_products") <= 24), "04. 1 - 2yr")
    .when((F.col("months_wam_to_other_products") > 24) & (F.col("months_wam_to_other_products") <= 36), "05. 2 - 3yr")
    .when((F.col("months_wam_to_other_products") > 36) & (F.col("months_wam_to_other_products") <= 60), "06. 3 - 5yr")
    .when((F.col("months_wam_to_other_products") > 60) & (F.col("months_wam_to_other_products") <= 84), "07. 5 - 7yr")
    .when((F.col("months_wam_to_other_products") > 84) & (F.col("months_wam_to_other_products") <=120), "08. 7 - 10yr")
    .when((F.col("months_wam_to_other_products") >120) & (F.col("months_wam_to_other_products") <=180), "09. 10 - 15yr")
    .when((F.col("months_wam_to_other_products") >180) & (F.col("months_wam_to_other_products") <=240), "10. 15 - 20yr")
    .otherwise("11. > 20yr")
)
mim_start_summary = mim_start.filter(
    F.col("months_wam_to_other_products") > 0
).groupBy(
    "months_wam_to_other_products_cat"
).agg(
    F.count("wam_client_id").alias("count"),
    F.avg("months_wam_to_other_products").alias("avg_months_wam_to_other_products")
).orderBy(
    "months_wam_to_other_products_cat"
)

# c. Customers starting with GP
gp_start = df_multiple_relationships.filter(
    df["earliest_gp_client_date"].isNotNull()
)
gp_start = gp_start.withColumn(
    "months_gp_to_other_products",
    F.months_between(
        F.least(df["first_inow_submit_date"], df["earliest_wam_client_date"]),
        df["earliest_gp_client_date"]
    )
).withColumn(
    "months_gp_to_other_products_cat",
    F.when(F.col("months_gp_to_other_products") <= 2, "01. <= 2mth")
    .when((F.col("months_gp_to_other_products") > 2) & (F.col("months_gp_to_other_products") <= 6), "02. 3 - 6mth")
    .when((F.col("months_gp_to_other_products") > 6) & (F.col("months_gp_to_other_products") <= 12), "03. 7 - 12mth")
    .when((F.col("months_gp_to_other_products") > 12) & (F.col("months_gp_to_other_products") <= 24), "04. 1 - 2yr")
    .when((F.col("months_gp_to_other_products") > 24) & (F.col("months_gp_to_other_products") <= 36), "05. 2 - 3yr")
    .when((F.col("months_gp_to_other_products") > 36) & (F.col("months_gp_to_other_products") <= 60), "06. 3 - 5yr")
    .when((F.col("months_gp_to_other_products") > 60) & (F.col("months_gp_to_other_products") <= 84), "07. 5 - 7yr")
    .when((F.col("months_gp_to_other_products") > 84) & (F.col("months_gp_to_other_products") <=120), "08. 7 - 10yr")
    .when((F.col("months_gp_to_other_products") >120) & (F.col("months_gp_to_other_products") <=180), "09. 10 - 15yr")
    .when((F.col("months_gp_to_other_products") >180) & (F.col("months_gp_to_other_products") <=240), "10. 15 - 20yr")
    .otherwise("11. > 20yr")
)
gp_start_summary = gp_start.filter(
    F.col("months_gp_to_other_products") > 0
).groupBy(
    "months_gp_to_other_products_cat"
).agg(
    F.count("gp_aum_usd").alias("count"),
    F.avg("months_gp_to_other_products").alias("avg_months_gp_to_other_products")
)

# 3) Create segments for MCV and age groups
mcv_segment_summary = df_multiple_relationships.groupBy(
    "mcv",
    "age_group"
).agg(
    F.count("lifeco_client_id").alias("total_customers")
).orderBy(
    F.desc("mcv"),
    "age_group"
)

# 4) Only lifeco customers
lifeco_customers = df.filter(
    df["earliest_wam_client_date"].isNull() &
    df["earliest_gp_client_date"].isNull()
)
lifeco_customers = lifeco_customers.withColumn(
    "number_of_insurance_prod",
    (F.when(F.col("saving_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("income_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("hospital_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("life_protection_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("critical_illness_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("investment_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("premium_waiver_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("disability_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("juvenile_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("accident_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("others_policy_cnt") > 0, 1).otherwise(0))
)
lifeco_customers = lifeco_customers.withColumn(
    "number_of_insurance_prod_cat",
    F.when(F.col("number_of_insurance_prod") == 0, "0. None")
    .when(F.col("number_of_insurance_prod") == 1, "1. Only 1")
    .when(F.col("number_of_insurance_prod").between(2, 3), "2. 2 - 3")
    .when(F.col("number_of_insurance_prod").between(4, 5), "3. 4 - 5")
    .when(F.col("number_of_insurance_prod").between(6, 7), "4. 6 - 7")
    .otherwise("5. 8+")
)

lifeco_summary = lifeco_customers.filter(
    F.col("number_of_insurance_prod_cat") != "0. None"
).groupBy(
    "number_of_insurance_prod_cat"
).agg(
    F.count("lifeco_client_id").alias("total_customers"),
    F.avg("inow_tenure_in_yr").alias("avg_tenure"), 
    F.sum(F.col("saving_policy_cnt") + F.col("income_policy_cnt") + F.col("hospital_policy_cnt") + F.col("life_protection_policy_cnt") + F.col("critical_illness_policy_cnt") + F.col("investment_policy_cnt") + F.col("premium_waiver_policy_cnt") + F.col("disability_policy_cnt") + F.col("juvenile_policy_cnt") + F.col("accident_policy_cnt") + F.col("others_policy_cnt")).alias("total_insurance_products"),
    F.sum("ape_usd").alias("total_ape")
).orderBy(
    "number_of_insurance_prod_cat"
)

# a. Same for customers with multiple products
lifeco_multiple_products = df.filter(
    (df["lifeco_ind"] == "Y") &
    (df["earliest_wam_client_date"].isNotNull() |
     df["earliest_gp_client_date"].isNotNull())
)
lifeco_multiple_products = lifeco_multiple_products.withColumn(
    "number_of_insurance_prod",
    (F.when(F.col("saving_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("income_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("hospital_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("life_protection_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("critical_illness_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("investment_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("premium_waiver_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("disability_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("juvenile_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("accident_policy_cnt") > 0, 1).otherwise(0) +
     F.when(F.col("others_policy_cnt") > 0, 1).otherwise(0))
).withColumn(
    "number_of_insurance_prod_cat",
    F.when(F.col("number_of_insurance_prod") == 0, "0. None")
    .when(F.col("number_of_insurance_prod") == 1, "1. Only 1")
    .when(F.col("number_of_insurance_prod").between(2, 3), "2. 2 - 3")
    .when(F.col("number_of_insurance_prod").between(4, 5), "3. 4 - 5")
    .when(F.col("number_of_insurance_prod").between(6, 7), "4. 6 - 7")
    .otherwise("5. 8+")
)
lifeco_multiple_summary = lifeco_multiple_products.groupBy(
    "number_of_insurance_prod_cat"
).agg(
    F.count("lifeco_client_id").alias("total_customers"),
    F.avg("inow_tenure_in_yr").alias("avg_tenure"), 
    F.sum(F.col("saving_policy_cnt") + F.col("income_policy_cnt") + F.col("hospital_policy_cnt") + F.col("life_protection_policy_cnt") + F.col("critical_illness_policy_cnt") + F.col("investment_policy_cnt") + F.col("premium_waiver_policy_cnt") + F.col("disability_policy_cnt") + F.col("juvenile_policy_cnt") + F.col("accident_policy_cnt") + F.col("others_policy_cnt")).alias("total_insurance_products"),
    F.sum(F.coalesce(F.col("gp_aum_usd"), F.lit(0)) + F.coalesce(F.col("wam_aum_usd"), F.lit(0))).alias("total_aum"),
    F.sum("ape_usd").alias("total_ape")
).orderBy(
    "number_of_insurance_prod_cat"
)

# 5) Only GP customers
gp_customers = df.filter(
    df["gp_aum_usd"].isNotNull()
)
gp_summary = gp_customers.filter(
    (df["lifeco_ind"] == "N") &
    df["wam_client_id"].isNull()
).groupBy().agg(
    F.count("lifeco_client_id").alias("total_customers"),
    F.avg("gp_relationship_duration").alias("avg_tenure"), 
    F.sum("gp_aum_usd").alias("total_aum")
)

# a. Same for customers with multiple products
gp_multiple_products = gp_customers.filter(
    (df["lifeco_ind"] == "Y") |
    df["wam_client_id"].isNotNull()
)
gp_multiple_summary = gp_multiple_products.groupBy().agg(
    F.count("lifeco_client_id").alias("total_customers"),
    F.avg("gp_relationship_duration").alias("avg_tenure"), 
    F.sum("gp_aum_usd").alias("total_aum"),
    F.sum("ape_usd").alias("total_ape")
)

# 6) Only MIM customers
mim_customers = df.filter(
    df["wam_client_id"].isNotNull()
)
mim_summary = mim_customers.withColumn(
    "tenure_in_years", 
    F.floor(F.datediff(F.current_date(), df["earliest_wam_client_date"]) / 365.25)
)
mim_summary = mim_summary.groupBy().agg(
    F.count("lifeco_client_id").alias("total_customers"),
    F.avg("tenure_in_years").alias("avg_tenure"), 
    F.sum("wam_aum_usd").alias("total_aum")
)

# a. Same for customers with multiple products
mim_multiple_products = mim_customers.filter(
    (df["lifeco_ind"] == "Y") |
    df["gp_aum_usd"].isNotNull()
)
mim_multiple_products = mim_multiple_products.withColumn(
    "tenure_in_years", 
    F.floor(F.datediff(F.current_date(), df["earliest_wam_client_date"]) / 365.25)
)
mim_multiple_summary = mim_multiple_products.groupBy().agg(
    F.count("lifeco_client_id").alias("total_customers"),
    F.avg("tenure_in_years").alias("avg_tenure"), 
    F.sum("wam_aum_usd").alias("total_aum"),
    F.sum("ape_usd").alias("total_ape")
)

# 7) Total AUM and APE by purchase sequence and number of products
purchase_sequence_summary = df_multiple_relationships.groupBy(
    "purchase_sequence"
).agg(
    F.count("lifeco_client_id").alias("total_customers"),
    F.sum(F.coalesce(F.col("gp_aum_usd"), F.lit(0)) + F.coalesce(F.col("wam_aum_usd"), F.lit(0))).alias("total_aum"), 
    F.sum(F.col("saving_policy_cnt") + F.col("income_policy_cnt") + F.col("hospital_policy_cnt") + F.col("life_protection_policy_cnt") + F.col("critical_illness_policy_cnt") + F.col("investment_policy_cnt") + F.col("premium_waiver_policy_cnt") + F.col("disability_policy_cnt") + F.col("juvenile_policy_cnt") + F.col("accident_policy_cnt") + F.col("others_policy_cnt")).alias("total_products"),
    F.sum("ape_usd").alias("total_ape")
)

df_multiple_relationships = df_multiple_relationships.join(
    lifeco_start.filter(
        F.col("months_lifeco_to_other_products") > 0
    ).select(
        "lifeco_client_id",
        "months_lifeco_to_other_products",
        "months_lifeco_to_other_products_cat"
    ),
    "lifeco_client_id",
    "left"
).join(
    mim_start.filter(
        F.col("months_wam_to_other_products") > 0
    ).select(
        "lifeco_client_id",
        "months_wam_to_other_products",
        "months_wam_to_other_products_cat"
    ),
    "lifeco_client_id",
    "left"
).join(
    gp_start.filter(
        F.col("months_gp_to_other_products") > 0
    ).select(
        "lifeco_client_id",
        "months_gp_to_other_products",
        "months_gp_to_other_products_cat"
    ),
    "lifeco_client_id",
    "left"
).join(
    lifeco_multiple_products.select(
        "lifeco_client_id",
        "number_of_insurance_prod_cat"
    ),
    "lifeco_client_id",
    "left"
).fillna(
    {
        "months_lifeco_to_other_products_cat": "00. None",
        "months_wam_to_other_products_cat": "00. None",
        "months_gp_to_other_products_cat": "00. None",
        "number_of_insurance_prod_cat": "0. None"
    }
)

# check_dup(df_multiple_relationships, "lifeco_client_id")

In [0]:
# Display results
print("2a. Customers start with LifeCo:")
lifeco_start_summary.display()
print("2b. Customers start with MIM:")
mim_start_summary.display()
print("2c. Customers start with GP:")
gp_start_summary.display()
print("3. Customer segments by MCV and Age group:")
mcv_segment_summary.display()
print("4. Customers holding ONLY LifeCo:")
lifeco_summary.display()
print("4a. Customers holding LifeCo + other:")
lifeco_multiple_summary.display()
print("5. Customers holding ONLY GP:")
gp_summary.display()
print("5a. Customers holding GP + other:")
gp_multiple_summary.display()
print("6. Customers holding ONLY MIM:")
mim_summary.display()
print("6a. Customers holding MIM + other:")
mim_multiple_summary.display()
print("7. Purchase sequence:")
purchase_sequence_summary.display()

In [0]:
df_multiple = df_multiple_relationships.toPandas()

In [0]:
df_multiple.to_csv("/dbfs/mnt/lab/vn/project/scratch/adhoc/GP_MIM/multiple_products.csv", index=False, header=True)